# RAG Retrieval Accuracy — Hybrid Score Fusion (6-d Ordinal Facet + Raw Dense)

For each trait, combines two **independent scores** — no vector fusion, no embedding
model called for facets:

```
score(q, t) = RAW_ALPHA  * dense_sim(q, t)          # raw-post embedding cosine
            + FACET_ALPHA * facet_cosine_6d(q, t)    # ordinal trait facet cosine
```

| Signal | Source | Dim |
|---|---|---|
| `dense_sim` | `embed(raw_post)` via rawpost index | 768 |
| `facet_cosine_6d` | `SIGNAL_TO_VALUE` encoding of the 6 trait facets | 6 |

The facet encoding maps GPT-4o-mini signals directly to numbers
(`high→+1`, `mod→+0.5`, `none→0`, `low→-1`) with no embedding model — same
as `build_hybrid_index.ipynb`.  Only the 6 facets relevant to each trait
are used (via `TRAIT_FACET_SLICES`), eliminating cross-trait noise.

Both scores are per-row min-max normalised to `[-1, 1]` before fusion
so the `FACET_ALPHA / RAW_ALPHA` weights are meaningful.

**Prerequisites:**
- `data/vector_db/essays_rawpost/` — raw-post FAISS index
- `data/vector_db/essays_dual/facet_vectors.npy` — training facet matrix
- `data/profile_db/essays_val/profile_store.jsonl` — val profiles

In [ ]:
import os, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import faiss

project_root = Path.cwd().resolve()
if not (project_root / "ptd_model").exists():
    project_root = (project_root / ".." / "..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)

## Configuration

In [ ]:
VAL_CSV        = str(project_root / "data/split/essays/val.csv")
VAL_PROFILE_DB = str(project_root / "data/profile_db/essays_val/profile_store.jsonl")
RAWPOST_DB     = str(project_root / "data/vector_db/essays_rawpost")
DUAL_DB        = str(project_root / "data/vector_db/essays_dual")
RES_DIR        = str(project_root / "result/retrieve_accuracy/hybrid_facet_6d")

K_VALUES    = [3, 5]
FACET_ALPHA = 0.8   # weight on 6-d ordinal facet score
RAW_ALPHA   = 0.2   # weight on raw-post dense score

TRAITS = {
    "cOPN": "Openness to Experience",
    "cCON": "Conscientiousness",
    "cEXT": "Extraversion",
    "cAGR": "Agreeableness",
    "cNEU": "Neuroticism",
}

val_df = pd.read_csv(VAL_CSV)
print(f"Val split : {len(val_df)} rows")

facet_npy = Path(DUAL_DB) / "facet_vectors.npy"
for label, p in [
    ("Val profiles",  VAL_PROFILE_DB),
    ("Rawpost DB",    RAWPOST_DB),
    ("facet_vectors", str(facet_npy)),
]:
    status = "OK" if Path(p).exists() else "MISSING"
    print(f"  {label:<16s}: [{status}]  {p}")

if not Path(RAWPOST_DB).exists():
    raise RuntimeError("Missing rawpost index.")
if not facet_npy.exists():
    raise RuntimeError("Missing facet_vectors.npy — rebuild dual index.")
if not Path(VAL_PROFILE_DB).exists():
    raise RuntimeError("Missing val profiles — run rag_retrieve_accuracy_profile.ipynb first.")

## Step 1 — Load training indexes and val profiles

In [ ]:
import json
from rag.profiler.store   import ProfileStore
from rag.profiler.prompts import FACETS
from rag.facet_vector     import load_facet_matrix, facet_vector, TRAIT_FACET_SLICES, FACET_ORDER

# Rawpost FAISS index — raw-text embeddings for training essays
rawpost_index = faiss.read_index(str(Path(RAWPOST_DB) / "vectors.faiss"))
rawpost_meta  = []
with open(str(Path(RAWPOST_DB) / "vectors_meta.jsonl"), encoding="utf-8") as f:
    for line in f:
        rawpost_meta.append(json.loads(line))
N_TRAIN = rawpost_index.ntotal
print(f"Rawpost index : {N_TRAIN} vectors, dim={rawpost_index.d}")

# Training facet matrix — built from profile store, aligned to rawpost_meta order
# facet_vectors.npy lives in essays_dual/ (side-product of build_index) but the
# data is profile-derived only. We re-align by user_id to be safe.
_facet_mat_raw   = load_facet_matrix(str(facet_npy))   # (N_DUAL, 30), dual-index order
_dual_meta       = []
with open(str(Path(DUAL_DB) / "vectors_meta.jsonl"), encoding="utf-8") as f:
    for line in f:
        _dual_meta.append(json.loads(line))
_uid_to_facet = {m["user_id"]: _facet_mat_raw[i] for i, m in enumerate(_dual_meta)}

# Build facet_mat in rawpost_meta order so facet_mat[i] matches rawpost_meta[i]
facet_mat = np.array(
    [_uid_to_facet.get(m["user_id"], np.zeros(30, dtype="float32")) for m in rawpost_meta],
    dtype="float32",
)
n_missing = int((facet_mat == 0).all(axis=1).sum())
print(f"Facet matrix  : {facet_mat.shape}  (missing profiles: {n_missing})")

# Val profiles
val_store = ProfileStore(VAL_PROFILE_DB)
val_store.load()
val_profiles = {
    int(e["user_id"].split("_")[1]): e
    for e in val_store.get_all() if e.get("valid")
}
print(f"Val profiles  : {len(val_profiles)} / {len(val_df)}")

## Step 2 — Embed val raw posts + compute val facet vectors

Only the embedding model call in the whole notebook — raw posts only, one batch.

In [ ]:
from rag.embedder     import _embed_single
from rag.facet_vector import facet_vector

N = len(val_df)

# Raw post embeddings (768-d) for val queries
print(f"Embedding {N} val raw posts ...")
t0 = time.time()
val_raw_embs = np.array(
    [_embed_single(str(row["text"])) for _, row in val_df.iterrows()],
    dtype="float32",
)
print(f"Done in {time.time()-t0:.1f}s  shape={val_raw_embs.shape}")

# Facet vectors (30-d) for val queries — ordinal encoding, no embedding model
val_fvecs = np.array(
    [facet_vector(val_profiles.get(i, {}), normalize=True) for i in range(N)],
    dtype="float32",
)
n_zero = int((np.linalg.norm(val_fvecs, axis=1) == 0).sum())
print(f"Val facet vectors: {val_fvecs.shape}  (zero/missing profiles: {n_zero})")

## Step 3 — Pre-compute all-pairs scores

- **Dense scores**: FAISS L2 search → convert to similarity → per-row min-max normalise
- **Facet scores** (per trait): cosine on the 6 trait-relevant facet dimensions → per-row min-max normalise

Both are computed once and stored in index order for fast per-(trait, k) evaluation.

In [ ]:
def per_row_minmax(M: np.ndarray) -> np.ndarray:
    """Rescale each row to [-1, 1] so different signal scales are comparable."""
    M    = np.array(M, dtype="float32")
    mins = M.min(axis=1, keepdims=True)
    maxs = M.max(axis=1, keepdims=True)
    rng  = np.where(maxs - mins > 0, maxs - mins, 1.0)
    return (2.0 * (M - mins) / rng - 1.0).astype("float32")


# Dense: search ALL training vectors, convert L2 -> similarity, put in index order
print(f"FAISS search: {N} queries x {N_TRAIN} candidates ...")
t0 = time.time()
all_distances, all_indices = rawpost_index.search(val_raw_embs, N_TRAIN)
# Convert L2 distance -> similarity (higher = better), reorder to index order
sim_ranked = 1.0 / (1.0 + all_distances)            # (N, N_TRAIN), in FAISS-rank order
dense_index = np.empty((N, N_TRAIN), dtype="float32")
for i in range(N):
    dense_index[i, all_indices[i]] = sim_ranked[i]   # put back in index position order
dense_norm = per_row_minmax(dense_index)             # (N, N_TRAIN), scaled to [-1, 1]
print(f"Done in {time.time()-t0:.1f}s  shape={dense_norm.shape}")

In [ ]:
# Facet cosine per trait — restricted to 6 trait-specific dimensions
print("Computing 6-d trait-restricted facet cosines ...")
t0 = time.time()
facet_scores = {}   # trait_code -> (N, N_TRAIN) normalised

for trait_code in TRAITS:
    idx = np.array(TRAIT_FACET_SLICES[trait_code], dtype=np.int64)  # 6 indices

    # Slice and re-normalise both sides to the 6-d subspace
    Vq = val_fvecs[:, idx].copy()                   # (N, 6)
    Vt = facet_mat[:, idx].copy()                   # (N_TRAIN, 6)

    q_norms = np.linalg.norm(Vq, axis=1, keepdims=True)
    t_norms = np.linalg.norm(Vt, axis=1, keepdims=True)
    Vq = Vq / np.where(q_norms > 0, q_norms, 1.0)
    Vt = Vt / np.where(t_norms > 0, t_norms, 1.0)

    raw_cos = Vq @ Vt.T                             # (N, N_TRAIN), cosine in [-1, 1]
    facet_scores[trait_code] = per_row_minmax(raw_cos)
    print(f"  {trait_code}: shape={facet_scores[trait_code].shape}  "
          f"mean={facet_scores[trait_code].mean():.3f}")

print(f"Done in {time.time()-t0:.1f}s")

## Step 4 — Evaluate retrieval accuracy per trait

Combined score: `RAW_ALPHA * dense_norm + FACET_ALPHA * facet_norm`

Rank descending, filter to entries with a label for this trait, take top-k.

In [ ]:
def normalize_label(val):
    s = str(val).strip().lower()
    if s in ("1", "1.0"): return "high"
    if s in ("0", "0.0"): return "low"
    return s


all_rows       = []
per_query_rows = []

print(f"FACET_ALPHA={FACET_ALPHA}  RAW_ALPHA={RAW_ALPHA}\n")

for k in K_VALUES:
    print(f"\n{'='*60}")
    print(f"  k = {k}")
    print(f"{'='*60}")

    for trait_code, trait_full in TRAITS.items():
        d_scores = dense_norm                       # (N, N_TRAIN)
        f_scores = facet_scores[trait_code]         # (N, N_TRAIN)
        scores   = RAW_ALPHA * d_scores + FACET_ALPHA * f_scores  # (N, N_TRAIN)

        match_rates, hits = [], []

        for i, (_, row) in enumerate(val_df.iterrows()):
            true_label = normalize_label(row[trait_code])
            if true_label not in ("high", "low"):
                continue

            ranked_idxs = np.argsort(scores[i])[::-1]

            filtered = []
            for idx in ranked_idxs:
                if len(filtered) >= k:
                    break
                meta = rawpost_meta[idx]
                if trait_full in meta.get("trait_labels", {}):
                    filtered.append(meta)

            n       = len(filtered)
            matches = sum(
                1 for r in filtered
                if r["trait_labels"][trait_full] == true_label
            )
            rate = matches / n if n > 0 else 0.0
            hit  = int(matches > 0)

            match_rates.append(rate)
            hits.append(hit)
            per_query_rows.append({
                "query_idx":   i,
                "trait":       trait_full,
                "k":           k,
                "true_label":  true_label,
                "n_retrieved": n,
                "match_count": matches,
                "match_rate":  rate,
                "hit":         hit,
            })

        mean_r = float(np.mean(match_rates))
        std_r  = float(np.std(match_rates))
        hit_r  = float(np.mean(hits))
        print(f"  {trait_full:<30s}  match={mean_r:.4f}+/-{std_r:.4f}  hit={hit_r:.4f}")

        all_rows.append({
            "trait":           trait_full,
            "k":               k,
            "n_queries":       len(match_rates),
            "mean_match_rate": mean_r,
            "std_match_rate":  std_r,
            "hit_rate":        hit_r,
        })

summary_df   = pd.DataFrame(all_rows)
per_query_df = pd.DataFrame(per_query_rows)
print("\nEvaluation complete.")

## Step 5 — Display & compare against baselines

In [ ]:
print("=== Mean match rate ===")
display(summary_df.pivot_table(index="trait", columns="k", values="mean_match_rate").round(4))

print("\n=== Hit rate ===")
display(summary_df.pivot_table(index="trait", columns="k", values="hit_rate").round(4))

In [ ]:
baseline_paths = {
    "profile":     project_root / "result/retrieve_accuracy/profile/accuracy_summary.csv",
    "rawpost":     project_root / "result/retrieve_accuracy/rawpost/accuracy_summary.csv",
    "full_dual":   project_root / "result/retrieve_accuracy/sliced_dual/accuracy_summary.csv",
    "facet_embed": project_root / "result/retrieve_accuracy/facet_embed/accuracy_summary.csv",
    "hybrid_facet":project_root / "result/retrieve_accuracy/hybrid_facet/accuracy_summary.csv",
}

for k_val in K_VALUES:
    print(f"\n=== Match rate comparison  (k={k_val}) ===")
    this = summary_df[summary_df["k"] == k_val].set_index("trait")["mean_match_rate"]
    compare = pd.DataFrame({"hybrid_6d": this})

    for name, path in baseline_paths.items():
        if not Path(path).exists():
            continue
        df = pd.read_csv(path)
        if "strategy" in df.columns:
            df = df[df["strategy"] == "full_dual"]
        if "gamma" in df.columns:
            # hybrid_facet notebook — pick gamma=0 row (pure dense baseline)
            df = df[df["gamma"] == 0.0]
        sub = df[df["k"] == k_val].set_index("trait")["mean_match_rate"]
        compare[name] = sub

    baseline_cols = [c for c in compare.columns if c != "hybrid_6d"]
    if baseline_cols:
        compare["best_baseline"] = compare[baseline_cols].max(axis=1)
        compare["delta_vs_best"] = compare["hybrid_6d"] - compare["best_baseline"]
    display(compare.round(4))

## Step 6 — Save

In [ ]:
os.makedirs(RES_DIR, exist_ok=True)

summary_path   = os.path.join(RES_DIR, "accuracy_summary.csv")
per_query_path = os.path.join(RES_DIR, "per_query_details.csv")

summary_df.to_csv(summary_path, index=False)
per_query_df.to_csv(per_query_path, index=False)

print(f"Saved summary   -> {summary_path}  ({len(summary_df)} rows)")
print(f"Saved per-query -> {per_query_path}  ({len(per_query_df)} rows)")